In [ ]:
# Asennetaan tarvittavat paketit
!pip install tpot

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 MB 3.9 MB/s eta 0:00:00
  Created wheel for stopit: filename=stopit-1.1.2-py3-none-any.whl size=11938 sha256=d33871fe5293be1cff1cc4c3c7884c03eb8424f4920f5316c5e6c3dc962181b2
  Stored in directory: /root/.cache/pip/wheels/af/f9/87/bf5b3d565c2a007b4dae9d8142dccc85a9f164e517062dd519
Successfully built stopit
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.3.2
    Uninstalling scikit-learn-1.3.2:
      Successfully uninstalled scikit-learn-1.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.

In [ ]:
# Import required libraries

# Ladataan tarvittavat kirjastot
from tpot import TPOTClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

In [ ]:
# Ladataan data

##bike_data seka bike_data_potentiaaliset on muokattu seuraavalla tavalla:
#sarakkeet nimetty samannimisiksi (esim. yearly income ja income nimetty samannimisiksi)
#puuttuvat arvot on korvattu keskiarvoisilla arvoilla
#bike_data_potentiaalisista on poistettu kaikki ylimääräiset sarakkeet (kuten nimi, sposti, osoite)
#eli sarakkeita on molemmissa tauluissa yhta monta.
#kaikki sarakkeet on laitettu samaan jarjestykseen

#Bike_data_potentiaaliset:ssa target arvoilla on annettu dummy-arvo -999
#tämä, koska -999 on ennalta-arvattavampi arvo kuin null tai tyhja tpotin
#käytön suhteen

# Alla oleva koodi ei toimi
#bike = pd.read_csv('bike_data.csv')
#bike.head()

# Yllä oleva koodi korvattu alla olevalla
from google.colab import drive
drive.mount('/content/drive')

import os
bike_file_path = '/content/drive/My Drive/Bike_data.csv'

bike = pd.read_csv(bike_file_path)
bike.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,ID,Income,Children,Cars,Age,target,Marital Status = Married,Marital Status = Single,Gender = Female,Gender = Male,...,Occupation = Management,Home Owner = Yes,Home Owner = No,Commute Distance = 0-1 Miles,Commute Distance = 2-5 Miles,Commute Distance = 5-10 Miles,Commute Distance = 1-2 Miles,Commute Distance = 10+ Miles,Region = Pacific,Region = North America
0,-1.396956,-0.519278,-0.558393,-1.291005,-0.192891,No,0.924355,-0.924355,1.017656,-1.017656,...,-0.457144,0.677786,-0.677786,1.315488,-0.439459,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622
1,0.774406,-0.841012,0.671548,-0.401883,-0.104813,No,0.924355,-0.924355,-0.981668,0.981668,...,-0.457144,0.677786,-0.677786,1.315488,-0.439459,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622
2,-1.082594,0.767657,1.901490,0.487239,1.392518,No,0.924355,-0.924355,-0.981668,0.981668,...,-0.457144,-1.473916,1.473916,-0.759414,2.273250,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622
3,0.825647,0.445923,-1.173364,-0.401883,-0.280970,Yes,-1.080754,1.080754,-0.981668,0.981668,...,-0.457144,0.677786,-0.677786,-0.759414,-0.439459,2.050396,-0.450739,-0.353178,2.050396,-1.015622
4,1.053050,-0.841012,-1.173364,-1.291005,-0.721361,Yes,-1.080754,1.080754,-0.981668,0.981668,...,-0.457144,-1.473916,1.473916,1.315488,-0.439459,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622


In [ ]:
#Katsotaan miten monta rivia ja saraketta datassa on
bike.shape

(1000, 28)

In [ ]:
#Katsotaan datan datatyyppi (joka nyt on pandas)
type(bike)

pandas.core.frame.DataFrame

In [ ]:
#muuteetaan target kentät yes ja no arvoiksi 1 ja 0
bike['target']=bike['target'].map({'No':0,'Yes':1})
bike.head()

,ID,Income,Children,Cars,Age,target,Marital Status = Married,Marital Status = Single,Gender = Female,Gender = Male,...,Occupation = Management,Home Owner = Yes,Home Owner = No,Commute Distance = 0-1 Miles,Commute Distance = 2-5 Miles,Commute Distance = 5-10 Miles,Commute Distance = 1-2 Miles,Commute Distance = 10+ Miles,Region = Pacific,Region = North America
0,-1.396956,-0.519278,-0.558393,-1.291005,-0.192891,0,0.924355,-0.924355,1.017656,-1.017656,...,-0.457144,0.677786,-0.677786,1.315488,-0.439459,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622
1,0.774406,-0.841012,0.671548,-0.401883,-0.104813,0,0.924355,-0.924355,-0.981668,0.981668,...,-0.457144,0.677786,-0.677786,1.315488,-0.439459,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622
2,-1.082594,0.767657,1.901490,0.487239,1.392518,0,0.924355,-0.924355,-0.981668,0.981668,...,-0.457144,-1.473916,1.473916,-0.759414,2.273250,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622
3,0.825647,0.445923,-1.173364,-0.401883,-0.280970,1,-1.080754,1.080754,-0.981668,0.981668,...,-0.457144,0.677786,-0.677786,-0.759414,-0.439459,2.050396,-0.450739,-0.353178,2.050396,-1.015622
4,1.053050,-0.841012,-1.173364,-1.291005,-0.721361,1,-1.080754,1.080754,-0.981668,0.981668,...,-0.457144,-1.473916,1.473916,1.315488,-0.439459,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622


In [ ]:
#Luodaan uusi bike data, jossa ei ole kenttiä ID ja target
bike_new = bike.drop(['ID','target'], axis=1)

In [ ]:
#Katsotaan bike_new:n datatyyppi, joka nyt on pandas
type(bike_new)

pandas.core.frame.DataFrame

In [ ]:
#muunnetaan .values komennolla bike_new tietyyppiin numpy
#uusi nimi bike_newest
#data pitää muuntaa numpyksi, tai tpotin koulutus ei onnistu
bike_newest = bike_new.values

In [ ]:
#katsotaan mitä bike_newest pitää sisällään
bike_newest[:][:]

array([[-0.51927812, -0.55839346, -1.29100539, ..., -0.35317776,
        -0.48722288, -1.01562188],
       [-0.84101178,  0.67154809, -0.40188322, ..., -0.35317776,
        -0.48722288, -1.01562188],
       [ 0.76765651,  1.90148964,  0.48723895, ..., -0.35317776,
        -0.48722288, -1.01562188],
       ...,
       [ 0.12418919,  0.05657731, -1.29100539, ..., -0.35317776,
        -0.48722288,  0.98363379],
       [ 1.41112382,  0.67154809,  1.37636112, ..., -0.35317776,
        -0.48722288,  0.98363379],
       [ 0.12418919,  0.67154809,  0.48723895, ...,  2.8286039 ,
        -0.48722288,  0.98363379]])

In [ ]:
#tarkistetaan, että tietotyyppi on numpy
type(bike_newest)

numpy.ndarray

In [ ]:
#Luodaan target datajoukko, jonka perusteella
#tekoälyn koulutus katsoo onko rivin arvo 0 vai 1

bike_target = bike['target'].values

#katsotaan bike_target sisältö
bike_target[:]

array([0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0,
       0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1,
       1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0,
       1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,

In [ ]:
#tarkistetaan, että bike_target:in tietotyyppi on numpy
#tekoälyn koulutus ei onnistu, jos tietotyyppi ei ole numpy, esim. panda
type(bike_target)

numpy.ndarray

In [ ]:
#tarkistetaan, että bike_target:illa on oikea määrä rivejä (1000)
#jolloin bike_target:in sisältö vastaa itse bike_data:n (ja new)
bike_target.shape

(1000,)

In [ ]:
#jaetaan koulutusjoukko bike_data_newest dataa käyttäen 75% ja 25% suhteella
X_train, X_test, y_train, y_test = train_test_split(bike_newest, bike_target, train_size=0.75, test_size=0.25)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((750, 26), (250, 26), (750,), (250,))

In [ ]:
# Koulutetaan tpot luokittelija, hyväksikäyttäen yllä tehtyä datan jakoa
# Huom: jos haluaa paremman luokittelijan, voi lisätä max_time_min parametria, esim = 10.
tpot = TPOTClassifier(verbosity=2, max_time_mins=2)
tpot.fit(X_train, y_train)
print(tpot.score(X_test, y_test))

Optimization Progress:   0%|          | 0/100 [00:00<?, ?pipeline/s]


2.02 minutes have elapsed. TPOT will close down.
TPOT closed during evaluation in one generation.


TPOT closed prematurely. Will use the current best pipeline.

Best pipeline: KNeighborsClassifier(SelectPercentile(input_matrix, percentile=52), n_neighbors=51, p=1, weights=distance)
0.672


In [ ]:
#exportataan tpot pipeline, mikäli sitä tarvitaan myöhemmin,
#esimerkiksi jos halutaan tarkastella tpot luokittelijan tarkkuutta
tpot.export('tpot_bike_pipeline.py')

In [ ]:
bike.to_csv('out.csv', sep=',')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

# NOTE: Make sure that the class is labeled 'target' in the data file
tpot_data = pd.read_csv('out.csv', sep=',', dtype=np.float64)
features = tpot_data.drop('target', axis=1).values
training_features, testing_features, training_target, testing_target = \
            train_test_split(features, tpot_data['target'].values, random_state=None)

# Average CV score on the training set was:0.6919713172437294
exported_pipeline = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False, interaction_only=False),
    GradientBoostingClassifier(learning_rate=0.1, max_depth=10, max_features=0.25,
                               min_samples_leaf=17, min_samples_split=8, n_estimators=100, subsample=1.0)
)

exported_pipeline.fit(training_features, training_target)
results = exported_pipeline.predict(testing_features)

In [ ]:
#Tarkistetaan miten tarkka meidän malli on
from sklearn.metrics import accuracy_score
accuracy_score(testing_target, results)

0.736

In [ ]:
#ladataan data jota ennusteteaan

# Alla oleva koodi ei toimi
#bike_sub = pd.read_csv('Bike_data_potentiaaliset.csv')
#bike_sub.head()

# Uusi koodi:
# Huom: Bike_data.csv on ladattu minun google driveen.

bike_sub_file_path = '/content/drive/My Drive/Bike_data.csv'

bike_sub = pd.read_csv(bike_sub_file_path)

bike.head()

,ID,Income,Children,Cars,Age,target,Marital Status = Married,Marital Status = Single,Gender = Female,Gender = Male,...,Occupation = Management,Home Owner = Yes,Home Owner = No,Commute Distance = 0-1 Miles,Commute Distance = 2-5 Miles,Commute Distance = 5-10 Miles,Commute Distance = 1-2 Miles,Commute Distance = 10+ Miles,Region = Pacific,Region = North America
0,-1.396956,-0.519278,-0.558393,-1.291005,-0.192891,0,0.924355,-0.924355,1.017656,-1.017656,...,-0.457144,0.677786,-0.677786,1.315488,-0.439459,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622
1,0.774406,-0.841012,0.671548,-0.401883,-0.104813,0,0.924355,-0.924355,-0.981668,0.981668,...,-0.457144,0.677786,-0.677786,1.315488,-0.439459,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622
2,-1.082594,0.767657,1.901490,0.487239,1.392518,0,0.924355,-0.924355,-0.981668,0.981668,...,-0.457144,-1.473916,1.473916,-0.759414,2.273250,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622
3,0.825647,0.445923,-1.173364,-0.401883,-0.280970,1,-1.080754,1.080754,-0.981668,0.981668,...,-0.457144,0.677786,-0.677786,-0.759414,-0.439459,2.050396,-0.450739,-0.353178,2.050396,-1.015622
4,1.053050,-0.841012,-1.173364,-1.291005,-0.721361,1,-1.080754,1.080754,-0.981668,0.981668,...,-0.457144,-1.473916,1.473916,1.315488,-0.439459,-0.487223,-0.450739,-0.353178,-0.487223,-1.015622


In [ ]:
#Poistetaan ennustettavasta datasta tiedot ID ja target
bike_sub_new = bike_sub.drop(['ID','target'], axis=1)

In [ ]:
#Tarkistetaan bike_sub_new:n tietotyyppi, joka on pandas
type(bike_sub_new)

pandas.core.frame.DataFrame

In [ ]:
#Muunnetaan bike_sub_new taulun tietotyyppi numpy:ksi
#Toimii myös pandas tietotyypillä
bike_sub_newest = bike_sub_new.values

In [ ]:
#tarkistetaan, että bike_sub_new on numpy
type(bike_sub_new)

pandas.core.frame.DataFrame

In [ ]:
#tarkistetaan, että bike_sub_new:issä on oikea määrä rivejä ja sarakkeita
bike_sub_new.shape

(1000, 26)

In [ ]:
#Tehdaan ennustus tpotin aiemmin kouluttamallamme luokittelijalla
submission = tpot.predict(bike_sub_new)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but SelectPercentile was fitted without feature names
  warnings.warn(


In [ ]:
#Katsotaan, miltä meidän ennustusdata näyttää
#(1 tarkoittaa, että ostaa pyörän, 0 että ei osta)
submission[:]

array([0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0,
       0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1,
       1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1,
       1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1,
       0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0,

In [ ]:
# Tehdään-csv tiedosto ennusteista
final = pd.DataFrame({'ID': bike_sub['ID'], 'target': submission})
final.to_csv('submission24.csv', index = False)

In [ ]:
#tarkistetaan osa 15kpl/78kpl ennustustuksista. Tässä näkyy vielä ostajan ID targetin vieressä.
final.head(15)

,ID,target
0,-1.396956,0
1,0.774406,0
2,-1.082594,1
3,0.825647,1
4,1.053050,1
5,-1.207890,0
6,1.497570,1
7,-0.112578,1
8,0.409364,0
9,-0.128287,1
